# H2-PRISM Baseline Models Evaluation (Person 2: Faisal)
Compares Logistic Regression, Random Forest, and XGBoost on scenario-level split.

## 1- Imports

In [2]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

from src.data import generate_mock_scenario
from src.features import extract_window_features
from src.fingerprint import extract_transient_fingerprint
from src.physics import compute_mass_balance_residual
from src.propagation import estimate_propagation_delay

## 2- Generate scenarios and split by scenario

In [3]:
scenario_types = [
    "normal",
    "leak",
    "operational_transient",
    "sensor_fault",
    "ambiguous",
]

scenario_features = []
for scenario_type in scenario_types:
    for seed in range(30):
        scenario = generate_mock_scenario(
            scenario_type=scenario_type,
            random_seed=seed,
        )
        scenario_id = f"{scenario_type}_{seed:03d}"
        features = extract_window_features(scenario)
        features["scenario_id"] = scenario_id
        features["label"] = scenario["label"].iloc[-1]
        scenario_features.append(features)

feature_table = pd.DataFrame(scenario_features)
train_scenarios, test_scenarios = train_test_split(
    feature_table["scenario_id"],
    test_size=0.25,
    random_state=42,
    stratify=feature_table["label"],
)

print(f"Scenarios: {len(feature_table)} total")
print(feature_table["label"].value_counts().sort_index())

Scenarios: 150 total
label
AMBIGUOUS                30
LEAK                     30
NORMAL                   30
OPERATIONAL_TRANSIENT    30
SENSOR_FAULT             30
Name: count, dtype: int64


In [4]:
excluded_columns = {"label", "scenario_id"}

feature_names = [
    column
    for column in feature_table.columns
    if column not in excluded_columns
]

feature_table[feature_names] = (
    feature_table[feature_names]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
)

train_table = feature_table[
    feature_table["scenario_id"].isin(train_scenarios)
].copy()

test_table = feature_table[
    feature_table["scenario_id"].isin(test_scenarios)
].copy()

X_train = train_table[feature_names]
X_test = test_table[feature_names]

y_train = train_table["label"]
y_test = test_table["label"]

## 3- Train and compare baseline models

In [5]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

evaluation_rows = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    balanced_accuracy = balanced_accuracy_score(y_test, predictions)
    evaluation_rows.append(
        {"model": model_name, "balanced_accuracy": balanced_accuracy}
    )

    print(f"\n{model_name} | balanced accuracy: {balanced_accuracy:.3f}")
    print(classification_report(y_test, predictions, zero_division=0))

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train_encoded)
xgb_predictions = label_encoder.inverse_transform(
    xgb_model.predict(X_test).astype(int)
)
xgb_balanced_accuracy = balanced_accuracy_score(y_test, xgb_predictions)
evaluation_rows.append(
    {"model": "XGBoost", "balanced_accuracy": xgb_balanced_accuracy}
)

print(f"\nXGBoost | balanced accuracy: {xgb_balanced_accuracy:.3f}")
print(classification_report(y_test, xgb_predictions, zero_division=0))

evaluation_table = pd.DataFrame(evaluation_rows).sort_values(
    "balanced_accuracy",
    ascending=False,
    ignore_index=True,
)
display(evaluation_table)


Logistic Regression | balanced accuracy: 1.000
                       precision    recall  f1-score   support

            AMBIGUOUS       1.00      1.00      1.00         8
                 LEAK       1.00      1.00      1.00         7
               NORMAL       1.00      1.00      1.00         8
OPERATIONAL_TRANSIENT       1.00      1.00      1.00         8
         SENSOR_FAULT       1.00      1.00      1.00         7

             accuracy                           1.00        38
            macro avg       1.00      1.00      1.00        38
         weighted avg       1.00      1.00      1.00        38


Random Forest | balanced accuracy: 1.000
                       precision    recall  f1-score   support

            AMBIGUOUS       1.00      1.00      1.00         8
                 LEAK       1.00      1.00      1.00         7
               NORMAL       1.00      1.00      1.00         8
OPERATIONAL_TRANSIENT       1.00      1.00      1.00         8
         SENSOR_FAULT   

,model,balanced_accuracy
0,Logistic Regression,1.0
1,Random Forest,1.0
2,XGBoost,1.0
